# Machine Learning: Classification and Regression using Scikit-learn

**Algorithms**
1. Random Forest Classification
2. Decision Tree Regression
3. Support Vector Regression (SVR)

**Dataset:** Diabetes dataset (built into scikit-learn)

> Note: The diabetes dataset has a continuous target (a quantitative disease-progression measure). It is suitable for regression, so it is used for Decision Tree Regression and SVR. For Random Forest Classification, the target is converted into two classes using the median of the target values. This is a teaching example; it does not make the dataset a clinical diagnostic tool.

This notebook uses built-in scikit-learn estimators and evaluation functions.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

## 2. Load and inspect the public dataset

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.describe())

## 3. Random Forest Classification

The original target is continuous. To demonstrate classification, label observations as:
- **0 (Lower progression):** target at or below the dataset median
- **1 (Higher progression):** target above the dataset median

The median is calculated from the full dataset here solely to define the illustrative label. In a real predictive workflow, any learned preprocessing or threshold selection should be determined using training data only.

In [ ]:
median_target = y.median()
y_class = (y > median_target).astype(int)

print("Median target:", median_target)
print("Class counts:")
print(y_class.value_counts().sort_index())

### Split the classification data

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print("Training samples:", X_train_c.shape[0])
print("Testing samples:", X_test_c.shape[0])

### Train Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train_c, y_train_c)

y_pred_c = rf_model.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("\nClassification report:")
print(classification_report(
    y_test_c, y_pred_c,
    target_names=["Lower progression", "Higher progression"],
    zero_division=0
))

cm = confusion_matrix(y_test_c, y_pred_c)
display(pd.DataFrame(
    cm,
    index=["Actual lower", "Actual higher"],
    columns=["Predicted lower", "Predicted higher"]
))

### Feature importance

In [ ]:
importances = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=True)

importances.plot(kind="barh", figsize=(8, 5))
plt.title("Random Forest: Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 4. Decision Tree Regression

Regression predicts a continuous numeric value. Here, the model predicts the original diabetes progression measure.

We evaluate the model using:
- **MAE:** average absolute prediction error (lower is better)
- **MSE:** average squared prediction error (lower is better)
- **RMSE:** square root of MSE, in target units (lower is better)
- **R²:** coefficient of determination (closer to 1 is generally better; it can be negative)

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree_model = DecisionTreeRegressor(
    max_depth=4,
    random_state=42
)
tree_model.fit(X_train_r, y_train_r)

y_pred_tree = tree_model.predict(X_test_r)

print("Decision Tree Regression")
print("MAE:", mean_absolute_error(y_test_r, y_pred_tree))
print("MSE:", mean_squared_error(y_test_r, y_pred_tree))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_tree)))
print("R²:", r2_score(y_test_r, y_pred_tree))

### Visualize the decision tree

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    max_depth=3
)
plt.title("Decision Tree Regressor (display limited to depth 3)")
plt.show()

## 5. Support Vector Regression (SVR)

SVR predicts continuous values. Feature scaling is important for SVM-based models, so `StandardScaler` is included in a pipeline. The scaler is fitted only on the training data.

In [ ]:
svr_model = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf", C=100, epsilon=10)
)
svr_model.fit(X_train_r, y_train_r)

y_pred_svr = svr_model.predict(X_test_r)

print("Support Vector Regression")
print("MAE:", mean_absolute_error(y_test_r, y_pred_svr))
print("MSE:", mean_squared_error(y_test_r, y_pred_svr))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_svr)))
print("R²:", r2_score(y_test_r, y_pred_svr))

## 6. Compare regression models

In [ ]:
results = pd.DataFrame({
    "Model": ["Decision Tree Regressor", "Support Vector Regressor"],
    "MAE": [
        mean_absolute_error(y_test_r, y_pred_tree),
        mean_absolute_error(y_test_r, y_pred_svr)
    ],
    "MSE": [
        mean_squared_error(y_test_r, y_pred_tree),
        mean_squared_error(y_test_r, y_pred_svr)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test_r, y_pred_tree)),
        np.sqrt(mean_squared_error(y_test_r, y_pred_svr))
    ],
    "R2": [
        r2_score(y_test_r, y_pred_tree),
        r2_score(y_test_r, y_pred_svr)
    ]
})
display(results)

### Actual vs predicted values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test_r, y_pred_tree, alpha=0.7)
axes[0].set_title("Decision Tree Regressor")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

axes[1].scatter(y_test_r, y_pred_svr, alpha=0.7)
axes[1].set_title("Support Vector Regressor")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

for ax in axes:
    low = min(y_test_r.min(), y_pred_tree.min(), y_pred_svr.min())
    high = max(y_test_r.max(), y_pred_tree.max(), y_pred_svr.max())
    ax.plot([low, high], [low, high], linestyle="--")
plt.tight_layout()
plt.show()

## 7. Conclusion

- **Random Forest Classifier** predicts one of two illustrative classes derived from the continuous target.
- **Decision Tree Regressor** predicts a continuous target using tree-based splits.
- **SVR** predicts a continuous target using a support-vector regression model. Scaling is included because SVR is sensitive to feature scales.
- Classification is evaluated with accuracy and a classification report. Regression is evaluated with MAE, MSE, RMSE and R².

The classification and regression tasks use separate target formats and separate train/test splits. The metrics across these task types are not directly comparable.